# Pyramid beam-splitter circuits
Runs the pyramid BS test for any range of layer counts `L`, using `nq = 2*L` qubits and a single excitation placed at the center qubit `L-1`.

**Pyramid schedule.** Layer `k` (for `k = 1..L`) applies `k` beam splitters on the consecutive qubit pairs starting at `center-(k-1)` and stepping by 2:

- layer 1: `BS(center, center+1)`
- layer 2: `BS(center-1, center)`, `BS(center+1, center+2)`
- layer k: `BS(center-(k-1)+2i, center-(k-1)+2i+1)` for `i = 0..k-1`

Barriers separate the initial X from layer 1 and each layer from the next.

Set `print_circuit = True` to draw every circuit; set `LAYERS = range(1, 8)` to reproduce the original seven notebooks.

In [ ]:
# =============================================================================
# User configuration — toggle sections on/off before "Run All".
# =============================================================================
# Imports, helper functions and backends always execute (everything depends on
# them). Each flag below guards a heavier or optional section.

RUN_TESTS            = True   # Unit tests for helpers (hamming_weight_hist, tvd, Bhattacharyya, convergence).
RUN_CONVERGENCE      = False  # Shot-convergence sweep — slow: many noisy runs per layer.
RUN_CIRCUITS         = True   # Main experiment: run pyramid circuits and collect counts + metrics.
RUN_DENSITY_MATRIX   = False   # Analytical noisy probabilities via density-matrix sim. Memory 16*4^nq bytes.
DENSITY_MATRIX_MAX_L = 6      # Skip density-matrix sim for L > this (memory-prohibitive). nq=12 => ~256 MB.
RUN_SUMMARY          = True   # Depth table and Bhattacharyya / Hamming-weight plots. Requires RUN_CIRCUITS.
SAVE_RESULTS         = True   # Persist `results` dict to JSON. Requires RUN_CIRCUITS.
RUN_VLQ_REAL         = True  # Submit each pyramid circuit to the real VLQ device at https://vlq.it4i.cz.
                              # Requires IQM_TOKEN env var. Real device time -- keep False unless you mean it.

print_circuit        = True   # Draw each pyramid circuit before running it. Set False for speed / less clutter.

# User input regarding running pyramids: 
# LAYERS = list(range(1, 9))                              # layer counts to sweep
# SHOTS_MANUAL = [1000, 1000, 2000, 2000, 4000, 4000, 8000, 8000]  # fallback for layers not convergence-tested
LAYERS = list(range(1, 5))                              # layer counts to sweep
SHOTS_MANUAL = [1000, 1000, 2000, 2000]  # fallback for layers not convergence-tested

# Results filename is derived automatically by `make_run_dir("pyramid_BS")` in the
# `run-dir` cell below -- artifacts land in Results/pyramid_BS_<date>/.


In [ ]:
# required versions
#!python --version # 3.11.13
#!pip show qiskit # 2.3.0
#!pip show iqm-client # 33.0.4

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from iqm.qiskit_iqm import transpile_to_IQM
from fake_vlq import FakeVLQ, FakeVLQ_small
from numpy import pi

from utils.density_matrix import exact_noisy_probabilities

# Shared experiment / metric helpers (moved out of this notebook so other
# experiments can reuse them).
from utils.metrics import (
    SBhattC, _sbc_dicts, _sbc_counts,
    hamming_weight_hist, tvd, max_abs_diff,
)
from utils.experiments import (
    qc_bs, build_pyramid, prepare_basis_state,
    analytical_probabilities, pyramid_statevector, pyramid_probabilities_analytical,
)
from utils.results_io import make_run_dir, save_run, load_run, latest_run


In [ ]:
# qc_bs(a, b, n) and build_pyramid(L) now live in utils.experiments
# (imported at the top of this notebook).


In [ ]:
# hamming_weight_hist(counts, nq) and analytical_probabilities(L) now live
# in utils.metrics / utils.experiments respectively.


In [ ]:
# pyramid_statevector and pyramid_probabilities_analytical now live in
# utils.experiments. Fixed bit-ordering convention to match Qiskit's
# little-endian Statevector (verified against analytical_probabilities up
# to L=6 -- agreement is machine precision).


In [ ]:
# `tvd`, `max_abs_diff`, `SBhattC` moved to utils.metrics. `find_convergent_shots`
# stays here because it's only used by the convergence-sweep cell below.

def find_convergent_shots(L, backend, transpile_fn=transpile,
                          start_shots=1024, threshold_tvd=0.05, max_doublings=5):
    """Auto-double shots until the Hamming-weight histogram converges.

    Compares normalized histograms at shots_low vs shots_high=2*shots_low.
    Returns (converged_shots, history) where converged_shots is the smallest
    shot count whose histogram is within threshold_tvd of the next doubling,
    or None if max_doublings is exceeded.
    """
    nq = 2 * L
    qc = build_pyramid(L)
    qc_t = transpile_fn(qc, backend)

    shots_low = start_shots
    counts_low = backend.run(qc_t, shots=shots_low).result().get_counts()
    prob_low = [h / shots_low for h in hamming_weight_hist(counts_low, nq)]

    history = []
    for _ in range(max_doublings):
        shots_high = shots_low * 2
        counts_high = backend.run(qc_t, shots=shots_high).result().get_counts()
        prob_high = [h / shots_high for h in hamming_weight_hist(counts_high, nq)]

        t = tvd(prob_low, prob_high)
        m = max_abs_diff(prob_low, prob_high)
        history.append({
            "shots_low": shots_low,
            "shots_high": shots_high,
            "tvd": t,
            "max_diff": m,
        })

        if t < threshold_tvd:
            return shots_low, history

        shots_low = shots_high
        prob_low = prob_high

    return None, history


In [7]:
# Compare the analytical single-particle statevector formula against the
# Qiskit Statevector simulation of the full pyramid circuit.
_TOL = 1e-10
print(f"{'L':>3}  {'nq':>4}  {'max |Δprob|':>13}  {'TVD':>10}  {'SBC':>10}")
print("-" * 50)
for L_test in range(1, 7):
    probs_qiskit, _ = analytical_probabilities(L_test)
    probs_formula  = pyramid_probabilities_analytical(L_test)

    keys = set(probs_qiskit) | set(probs_formula)
    p = [probs_qiskit.get(k, 0.0) for k in keys]
    q = [probs_formula.get(k, 0.0) for k in keys]

    max_abs = max(abs(pi - qi) for pi, qi in zip(p, q))
    t       = tvd(p, q)
    fid     = SBhattC(p, q)

    flag = "OK" if max_abs < _TOL else "DIFFERS"
    print(f"{L_test:>3}  {2*L_test:>4}  {max_abs:>13.3e}  {t:>10.3e}  {fid:>10.6f}  {flag}")

  L    nq    max |Δprob|         TVD         SBC
--------------------------------------------------
  1     2      1.110e-16   1.110e-16    1.000000  OK
  2     4      3.331e-16   5.551e-16    1.000000  OK
  3     6      5.000e-01   5.000e-01    0.250000  DIFFERS
  4     8      5.000e-01   5.000e-01    0.562500  DIFFERS
  5    10      3.750e-01   6.250e-01    0.390625  DIFFERS
  6    12      2.500e-01   3.750e-01    0.765625  DIFFERS


In [8]:
backendAerSim = AerSimulator()
backendFakeVLQ = FakeVLQ()

# Smaller fake backend for density-matrix simulation: 12 best-performing
# VLQ qubits + CR1 (13 total) -- largest that fits density-matrix sim under
# the default 15-qubit cap (1 GB memory). Shares the VLQ calibration.
# `DENSITY_MATRIX_MAX_L=6` corresponds to nq=12.
backendFakeVLQ_small = FakeVLQ_small(n_qubits=12)

In [ ]:
# Connection to the real VLQ device (https://vlq.it4i.cz). Opt-in via RUN_VLQ_REAL.
# Authentication: relies on the IQM_TOKEN environment variable; see VLQ_experiments/5h.ipynb
# for the reference pattern. On failure (no token, unreachable server, ...) we set
# `backendVLQreal = None` and the sweep cell skips the real-device branch gracefully.
#
# `use_metrics=True`: backend pulls calibration data + quality metrics into the
# transpilation Target so `optimization_level=3` picks noise-aware layouts via VF2.
backendVLQreal = None
if RUN_VLQ_REAL:
    from iqm.qiskit_iqm import IQMProvider
    _VLQ_REAL_URL = "https://vlq.it4i.cz"
    try:
        _provider = IQMProvider(_VLQ_REAL_URL)
        backendVLQreal = _provider.get_backend(use_metrics=True)
        print(f"Connected to {_VLQ_REAL_URL}")
        print(f"  qubits: {backendVLQreal.architecture.qubits}")
        print(f"  gates:  {list(backendVLQreal.architecture.gates.keys())}")
    except Exception as e:
        backendVLQreal = None
        print(f"RUN_VLQ_REAL=True but connection failed: {e}")
        print("Real-device sweep will be skipped. Check IQM_TOKEN and network.")
else:
    print("RUN_VLQ_REAL=False -- skipping real-device backend setup.")


In [ ]:
# All plots and the persist cell write into a single dated run folder under Results/.
RUN_DIR = make_run_dir("pyramid_BS")
print(f"Saving this run's artifacts into: {RUN_DIR}")


In [11]:
def _run_noisefree(qc, shots):
    qct = transpile(qc, backendAerSim)
    return backendAerSim.run(qct, shots=shots).result().get_counts()

def test_hamming_weight_hist():
    """Validate hamming_weight_hist on circuits with known output distributions."""
    from math import comb, sqrt

    nq = 4
    shots = 2000

    # 1) |0...0>: every shot has Hamming weight 0.
    qc = QuantumCircuit(nq); qc.measure_all()
    hist = hamming_weight_hist(_run_noisefree(qc, shots), nq)
    assert hist == [shots, 0, 0, 0, 0], f"|0...0>: {hist}"
    print(f"  |0...0>         : hist={hist}  OK")

    # 2) |1...1>: every shot has Hamming weight nq.
    qc = QuantumCircuit(nq)
    for q in range(nq):
        qc.x(q)
    qc.measure_all()
    hist = hamming_weight_hist(_run_noisefree(qc, shots), nq)
    expected = [0] * (nq + 1); expected[nq] = shots
    assert hist == expected, f"|1...1>: {hist}"
    print(f"  |1...1>         : hist={hist}  OK")

    # 3) Single X: every shot has Hamming weight 1.
    qc = QuantumCircuit(nq); qc.x(1); qc.measure_all()
    hist = hamming_weight_hist(_run_noisefree(qc, shots), nq)
    expected = [0] * (nq + 1); expected[1] = shots
    assert hist == expected, f"single X: {hist}"
    print(f"  single X        : hist={hist}  OK")

    # 4) H on every qubit: uniform over {0,1}^nq => Binomial(nq, 1/2) on Hamming weight.
    qc = QuantumCircuit(nq)
    for q in range(nq):
        qc.h(q)
    qc.measure_all()
    hist = hamming_weight_hist(_run_noisefree(qc, shots), nq)
    assert sum(hist) == shots
    for k in range(nq + 1):
        p = comb(nq, k) / 2 ** nq
        mu = shots * p
        sigma = sqrt(shots * p * (1 - p)) if 0 < p < 1 else 0.0
        tol = 5 * sigma + 1
        assert abs(hist[k] - mu) <= tol, f"H^n bin k={k}: got {hist[k]}, expected {mu:.1f}+/-{sigma:.1f}"
    print(f"  H^{nq}            : hist={hist}  (matches Binomial({nq},1/2)*{shots})  OK")

    # 5) Pyramid is particle-number preserving => noise-free sim must give all weight-1.
    for L_test in (1, 2, 3):
        qc = build_pyramid(L_test)
        hist = hamming_weight_hist(_run_noisefree(qc, shots), 2 * L_test)
        expected = [0] * (2 * L_test + 1); expected[1] = shots
        assert hist == expected, f"pyramid L={L_test}: {hist}"
        print(f"  pyramid L={L_test}     : hist={hist}  (all weight-1 -- BS conserves particles)  OK")

    print("All hamming_weight_hist tests passed.")

def test_tvd_and_max_abs_diff():
    """Validate tvd and max_abs_diff on known distribution pairs."""
    # Identical distributions => both 0
    p = [0.25, 0.5, 0.25]
    assert tvd(p, p) == 0.0
    assert max_abs_diff(p, p) == 0.0
    print("  identical       : tvd=0, max_diff=0  OK")

    # Disjoint distributions => TVD=1, max_diff=1
    assert tvd([1, 0], [0, 1]) == 1.0
    assert max_abs_diff([1, 0], [0, 1]) == 1.0
    print("  disjoint        : tvd=1, max_diff=1  OK")

    # Known intermediate
    t = tvd([0.5, 0.5], [0.25, 0.75])
    m = max_abs_diff([0.5, 0.5], [0.25, 0.75])
    assert abs(t - 0.25) < 1e-12 and abs(m - 0.25) < 1e-12
    print(f"  [.5,.5] vs [.25,.75]: tvd={t}, max_diff={m}  OK")

    print("All tvd / max_abs_diff tests passed.")

def test_SBhattC():
    """Validate SBhattC on known distribution pairs."""
    # Identical distributions => F = 1
    p = [0.25, 0.5, 0.25]
    f = SBhattC(p, p)
    assert abs(f - 1.0) < 1e-12, f"identical: got {f}"
    print(f"  identical       : F={f:.6f}  OK")

    # Disjoint distributions => F = 0
    f = SBhattC([1, 0], [0, 1])
    assert f == 0.0, f"disjoint: got {f}"
    print(f"  disjoint        : F={f:.6f}  OK")

    # Known intermediate: [0.5, 0.5] vs [0.25, 0.75]
    # BC = sqrt(0.5*0.25) + sqrt(0.5*0.75) = sqrt(0.125) + sqrt(0.375)
    # F = BC^2
    from math import sqrt as msqrt
    bc = msqrt(0.125) + msqrt(0.375)
    expected_f = bc ** 2
    f = SBhattC([0.5, 0.5], [0.25, 0.75])
    assert abs(f - expected_f) < 1e-12, f"intermediate: got {f}, expected {expected_f}"
    print(f"  [.5,.5] vs [.25,.75]: F={f:.6f} (expected {expected_f:.6f})  OK")

    # Delta vs uniform: p=[1,0,0], q=[1/3,1/3,1/3] => BC = sqrt(1/3), F = 1/3
    f = SBhattC([1, 0, 0], [1/3, 1/3, 1/3])
    assert abs(f - 1/3) < 1e-12, f"delta vs uniform: got {f}"
    print(f"  delta vs uniform: F={f:.6f} (expected {1/3:.6f})  OK")

    print("All SBhattC tests passed.")

def test_find_convergent_shots():
    """Validate convergence on both deterministic (noiseless) and stochastic (noisy) circuits."""

    # --- Noiseless: deterministic sanity checks ---
    for L_test in [1, 2, 3]:
        shots_conv, history = find_convergent_shots(
            L_test, backendAerSim, start_shots=64, threshold_tvd=0.01)
        assert shots_conv == 64, (
            f"noiseless L={L_test}: deterministic histogram should converge at start_shots, got {shots_conv}")
        assert history[0]["tvd"] == 0.0, (
            f"noiseless L={L_test}: TVD should be exactly 0 (deterministic)")
        print(f"  noiseless L={L_test}: converged at {shots_conv} shots, "
              f"TVD={history[0]['tvd']:.5f} (deterministic)  OK")

    # --- Noisy backend: actual convergence behaviour ---
    for L_test in [1, 2, 3]:
        shots_conv, history = find_convergent_shots(
            L_test, backendFakeVLQ, transpile_fn=transpile_to_IQM,
            start_shots=1024, threshold_tvd=0.05, max_doublings=5)
        assert shots_conv is not None, f"noisy L={L_test} should converge within 5 doublings"
        assert history[0]["tvd"] > 0, f"noisy L={L_test}: first-step TVD should be > 0"
        for step in history:
            status = "PASS" if step["tvd"] < 0.05 else "    "
            print(f"    {step['shots_low']:>7} vs {step['shots_high']:>7}  "
                  f"TVD={step['tvd']:.5f}  max_diff={step['max_diff']:.5f}  {status}")
        print(f"  noisy L={L_test}: converged at {shots_conv} shots, steps={len(history)}  OK")

    print("All find_convergent_shots tests passed.")

if RUN_TESTS:
    test_hamming_weight_hist()
    print()
    test_tvd_and_max_abs_diff()
    print()
    test_SBhattC()
    print()
    test_find_convergent_shots()
else:
    print("RUN_TESTS=False -- skipping unit tests.")

  |0...0>         : hist=[2000, 0, 0, 0, 0]  OK
  |1...1>         : hist=[0, 0, 0, 0, 2000]  OK
  single X        : hist=[0, 2000, 0, 0, 0]  OK
  H^4            : hist=[139, 494, 760, 478, 129]  (matches Binomial(4,1/2)*2000)  OK
  pyramid L=1     : hist=[0, 2000, 0]  (all weight-1 -- BS conserves particles)  OK
  pyramid L=2     : hist=[0, 2000, 0, 0, 0]  (all weight-1 -- BS conserves particles)  OK
  pyramid L=3     : hist=[0, 2000, 0, 0, 0, 0, 0]  (all weight-1 -- BS conserves particles)  OK
All hamming_weight_hist tests passed.

  identical       : tvd=0, max_diff=0  OK
  disjoint        : tvd=1, max_diff=1  OK
  [.5,.5] vs [.25,.75]: tvd=0.25, max_diff=0.25  OK
All tvd / max_abs_diff tests passed.

  identical       : F=1.000000  OK
  disjoint        : F=0.000000  OK
  [.5,.5] vs [.25,.75]: F=0.933013 (expected 0.933013)  OK
  delta vs uniform: F=0.333333 (expected 0.333333)  OK
All SBhattC tests passed.

  noiseless L=1: converged at 64 shots, TVD=0.00000 (deterministic)  OK
  no

In [12]:
if RUN_CONVERGENCE:
    CONVERGENCE_LAYERS = [1, 2, 3, 4, 5, 6]   # layers to test (extend as needed; higher layers are slow)
    THRESHOLD_TVD = 0.001
    START_SHOTS = 1024

    converged_shots = {}

    for L in CONVERGENCE_LAYERS:
        print(f"\n--- Convergence test: L={L} (nq={2*L}), threshold_tvd={THRESHOLD_TVD} ---")
        shots_conv, history = find_convergent_shots(
            L, backendFakeVLQ,
            start_shots=START_SHOTS, threshold_tvd=THRESHOLD_TVD)
        converged_shots[L] = shots_conv
        for step in history:
            status = "PASS" if step["tvd"] < THRESHOLD_TVD else "    "
            print(f"  {step['shots_low']:>7} vs {step['shots_high']:>7}  "
                  f"TVD={step['tvd']:.5f}  max_diff={step['max_diff']:.5f}  {status}")
        if shots_conv is not None:
            print(f"  => converged at {shots_conv} shots")
        else:
            print(f"  => did NOT converge after {len(history)} doublings")

    print(f"\nConverged shots: {converged_shots}")
    print(f"\nCopy-paste for SHOTS_MANUAL:")
    print(f"SHOTS_MANUAL = {[converged_shots[L] for L in CONVERGENCE_LAYERS]}")
else:
    print("RUN_CONVERGENCE=False -- skipping shot-convergence sweep.")

RUN_CONVERGENCE=False -- skipping shot-convergence sweep.


In [ ]:
if RUN_CIRCUITS:
    # LAYERS = list(range(1, 9))                              # layer counts to sweep
    # SHOTS_MANUAL = [1000, 1000, 2000, 2000, 4000, 4000, 8000, 8000]  # fallback for layers not convergence-tested
    limit_for_plotting = 15                                 # only show bitstrings with more than this many counts

    # # Use converged shots where available, fall back to manual values otherwise.
    # SHOTS_LIST = [converged_shots.get(L, SHOTS_MANUAL[i]) for i, L in enumerate(LAYERS)]
    # print(f"Shots per layer: {dict(zip(LAYERS, SHOTS_LIST))}\n")

    SHOTS_LIST = SHOTS_MANUAL

    results = {}

    for L, shots in zip(LAYERS, SHOTS_LIST):
        nq = 2 * L
        print(f"\n=== Pyramid with {L} layer(s), {nq} qubits, shots={shots} ===")
        qc_gbs = build_pyramid(L)

        if print_circuit:
            display(qc_gbs.draw(output='mpl', fold=-1))

        qc_t_AerSim = transpile(qc_gbs, backendAerSim)

        # Order matters: transpile to real VLQ FIRST (with optimization_level=3, which
        # is what activates the noise-aware VF2 layout passes that consume the
        # calibration metrics loaded via `use_metrics=True`). Capture the chosen
        # physical-qubit layout, then pin FakeVLQ to the same qubits so the noise
        # channels driving the two simulations match the hardware run apples-to-apples.
        counts_vlq_real = None
        qc_t_VLQreal = None
        real_layout = None  # list of physical qubit indices selected on the real device
        if RUN_VLQ_REAL and backendVLQreal is not None:
            try:
                qc_t_VLQreal = transpile_to_IQM(qc_gbs, backendVLQreal, optimization_level=3)
                real_layout = qc_t_VLQreal.layout.initial_index_layout(filter_ancillas=True)
                print(f"  Real VLQ layout (physical qubits): {real_layout}")
                print(f"  Submitting to real VLQ ({shots} shots, depth={qc_t_VLQreal.depth()})...")
                job_real = backendVLQreal.run(qc_t_VLQreal, shots=shots)
                counts_vlq_real = job_real.result().get_counts()
                print(f"  Real VLQ run finished. {len(counts_vlq_real)} unique bitstrings.")
            except Exception as e:
                counts_vlq_real = None
                print(f"  Real VLQ run failed for L={L}: {e}")

        # FakeVLQ -- pin to the real-device layout when available so the noisy sim
        # uses the same physical qubits (and hence the same per-qubit noise parameters)
        # as the hardware run. `optimization_level=3` either way.
        if real_layout is not None:
            qc_t_VLQ = transpile_to_IQM(qc_gbs, backendFakeVLQ,
                                        initial_layout=real_layout,
                                        optimization_level=3)
            print(f"  FakeVLQ pinned to real-device layout: {real_layout}")
        else:
            qc_t_VLQ = transpile_to_IQM(qc_gbs, backendFakeVLQ, optimization_level=3)

        counts_vlq = backendFakeVLQ.run(qc_t_VLQ, shots=shots).result().get_counts()
        counts_aer = backendAerSim.run(qc_t_AerSim, shots=shots).result().get_counts()

        # Exact analytical probabilities from statevector (noise-free)
        probs_exact, hw_prob_exact = analytical_probabilities(L)
        # probs_exact = exact probability distribution 
        # hw_prob_exact = exact Hamming distances 

        hw_noisy = hamming_weight_hist(counts_vlq, nq) # Hamming distances for simulations on FakeVLQ
        hw_clean = hamming_weight_hist(counts_aer, nq) # Hamming distances for simulations without noise 
        pw_noisy = [h / shots for h in hw_noisy]    # normalized Hamming distances for simulations on FakeVLQ
        pw_clean = [h / shots for h in hw_clean]    # normalized Hamming distances for simulations without noise

        sbc_vs_aersim = SBhattC(pw_noisy, pw_clean)   # squared Bhattacharyya coefficient (SBC) of Hamming distances from FakeVLQ and noiseless simulations
        sbc_vs_exact = SBhattC(pw_noisy, hw_prob_exact) # squared Bhattacharyya coefficient (SBC) of Hamming distances from FakeVLQ and analytical solution

        # squared Bhattacharyya coefficient (SBC) on full bitstring distribution (not just Hamming weight)
        all_keys = set(counts_vlq.keys()) | set(probs_exact.keys())
        p_noisy_full = {k: counts_vlq.get(k, 0) / shots for k in all_keys}
        p_exact_full = {k: probs_exact.get(k, 0.0) for k in all_keys}
        ordered_keys = sorted(all_keys)
        sbc_bitstring_vs_exact = SBhattC(
            [p_noisy_full[k] for k in ordered_keys],
            [p_exact_full[k] for k in ordered_keys])

        # Exact *noisy* probabilities via density-matrix simulation (exact limit of backend.run).
        # Uses `backendFakeVLQ_small` (12 best VLQ qubits + CR1 = 13 total) because the full 24-qubit
        # VLQ would need 4**25 ~ 10**15 density-matrix entries. The small backend reuses the VLQ
        # calibration for its qubits, so shot-based and DM sim are on the same noise family -- they
        # just differ in which physical qubits are active. For layers that exceed what the small
        # backend can route (nq > 12), the guard trips and the DM block is skipped.
        probs_dm = None 
        hw_dm = None 
        sbc_hw_dm_vs_exact = None # squared Bhattacharyya coefficient (SBC) of Hamming distances from density matrix and analytical solution
        sbc_bs_dm_vs_exact = None # squared Bhattacharyya coefficient (SBC) of probability dist. from density matrix and analytical solution
        sbc_bs_noisy_vs_dm = None # squared Bhattacharyya coefficient (SBC) of probability dist. from density matrix and FakeVLQ simulation
        if RUN_DENSITY_MATRIX and L <= DENSITY_MATRIX_MAX_L:
            try:
                probs_dm_raw = exact_noisy_probabilities(qc_gbs, backendFakeVLQ_small)
                # Marginalise over ancilla qubits: keep only the rightmost nq bits
                # (Qiskit little-endian; transpile pads extra qubits on the left).
                probs_dm = {}
                for bs, p in probs_dm_raw.items():
                    key = bs[-nq:] if len(bs) >= nq else bs.zfill(nq)
                    probs_dm[key] = probs_dm.get(key, 0.0) + p
                hw_dm = [0.0] * (nq + 1)
                for bs, p in probs_dm.items():
                    hw_dm[bs.count("1")] += p

                sbc_hw_dm_vs_exact = SBhattC(hw_dm, hw_prob_exact)
                ordered_keys_exact = sorted(set(probs_dm.keys()) | set(probs_exact.keys()))
                sbc_bs_dm_vs_exact = SBhattC(
                    [probs_dm.get(k, 0.0)    for k in ordered_keys_exact],
                    [probs_exact.get(k, 0.0) for k in ordered_keys_exact])

                ordered_keys_dm = sorted(set(p_noisy_full.keys()) | set(probs_dm.keys()))
                sbc_bs_noisy_vs_dm = SBhattC(
                    [p_noisy_full.get(k, 0.0) for k in ordered_keys_dm],
                    [probs_dm.get(k, 0.0)     for k in ordered_keys_dm])
            except ValueError as e:
                print(f"  [density-matrix sim skipped: {e}]")
        elif RUN_DENSITY_MATRIX:
            print(f"  [density-matrix sim skipped: L={L} > DENSITY_MATRIX_MAX_L={DENSITY_MATRIX_MAX_L}]")

        # Real-VLQ SBC metrics (None if the real-device run was skipped).
        sbc_hw_vlqreal_vs_exact = None
        sbc_bs_vlqreal_vs_exact = None
        sbc_bs_vlqreal_vs_fakevlq = None
        if counts_vlq_real is not None:
            hw_vlqreal = hamming_weight_hist(counts_vlq_real, nq)
            pw_vlqreal = [h / shots for h in hw_vlqreal]
            sbc_hw_vlqreal_vs_exact = SBhattC(pw_vlqreal, hw_prob_exact)

            ordered_keys_real_exact = sorted(set(counts_vlq_real.keys()) | set(probs_exact.keys()))
            sbc_bs_vlqreal_vs_exact = SBhattC(
                [counts_vlq_real.get(k, 0) / shots for k in ordered_keys_real_exact],
                [probs_exact.get(k, 0.0)            for k in ordered_keys_real_exact])

            ordered_keys_real_fake = sorted(set(counts_vlq_real.keys()) | set(counts_vlq.keys()))
            sbc_bs_vlqreal_vs_fakevlq = SBhattC(
                [counts_vlq_real.get(k, 0) / shots for k in ordered_keys_real_fake],
                [counts_vlq.get(k, 0)       / shots for k in ordered_keys_real_fake])

            print(f"  SBC of HWs: VLQ_real vs analytical solution         : {sbc_hw_vlqreal_vs_exact:.6f}")
            print(f"  SBC of bitstrings: VLQ_real vs analytical solution  : {sbc_bs_vlqreal_vs_exact:.6f}")
            print(f"  SBC of bitstrings: VLQ_real vs FakeVLQ              : {sbc_bs_vlqreal_vs_fakevlq:.6f}")

        results[L] = {
            "L": L,
            "nq": nq,
            "shots": shots,
            "counts_noisy": dict(counts_vlq),
            "counts_noisefree": dict(counts_aer),
            "counts_VLQreal": dict(counts_vlq_real) if counts_vlq_real is not None else None,
            "probs_exact": probs_exact,
            "probs_exact_noisy": probs_dm,
            "metrics": {
                "hamming_weight_hist_noisy": hw_noisy,
                "hamming_weight_hist_noisefree": hw_clean,
                "hamming_weight_prob_noisy": pw_noisy,
                "hamming_weight_prob_noisefree": pw_clean,
                "hamming_weight_prob_exact": hw_prob_exact,
                "hamming_weight_prob_exact_noisy": hw_dm,
                "SBC_HW_FakeQVL_noiseless": sbc_vs_aersim, # squared Bhattacharyya coefficient (SBC) of Hamming distances from FakeVLQ and noiseless simulations
                "SBC_HW_FakeVLQ_vs_analytical": sbc_vs_exact, # squared Bhattacharyya coefficient (SBC) of Hamming distances from FakeVLQ and analytical solution
                "SBC_BS_FakeVLQ_vs_analytical": sbc_bitstring_vs_exact,
                "SBC_HW_DM_vs_analytical": sbc_hw_dm_vs_exact,
                "SBC_BS_DM_vs_analytical": sbc_bs_dm_vs_exact,
                "SBC_BS_FakeVLQ_vs_DM": sbc_bs_noisy_vs_dm,
                "SBC_HW_VLQreal_vs_analytical": sbc_hw_vlqreal_vs_exact,
                "SBC_BS_VLQreal_vs_analytical": sbc_bs_vlqreal_vs_exact,
                "SBC_BS_VLQreal_vs_FakeVLQ":     sbc_bs_vlqreal_vs_fakevlq,
            },        
            "circuit": {
                "depth_original": qc_gbs.depth(),
                "depth_transpiled_AerSim": qc_t_AerSim.depth(),
                "depth_transpiled_VLQ": qc_t_VLQ.depth(),
                "depth_transpiled_VLQreal": qc_t_VLQreal.depth() if qc_t_VLQreal is not None else None,
                "real_layout": list(real_layout) if real_layout is not None else None,
            },
        }

        print(f"Hamming-weight hist (k=0..{nq}):")
        print(f"  noisy (counts)      : {hw_noisy}")
        print(f"  noisy (prob)        : {[round(p, 4) for p in pw_noisy]}")
        print(f"  noise-free          : {hw_clean}")
        print(f"  exact    (analyt.)  : {[round(p, 6) for p in hw_prob_exact]}")
        if hw_dm is not None:
            print(f"  exact noisy (DM)    : {[round(p, 6) for p in hw_dm]}")
        print(f"  SBC of HWs: Fake_VLQ vs AerSim                      : {  sbc_vs_aersim:.6f}")
        print(f"  SBC of HWs: Fake_VLQ vs analytical solution         : {sbc_vs_exact:.6f}")
        print(f"  SBC of bitstrings: Fake_VLQ vs analytical solution  : {sbc_bitstring_vs_exact:.6f}")
        if sbc_hw_dm_vs_exact is not None:
            print(f"  SBC of HWs: DM    vs analytical solution      : {sbc_hw_dm_vs_exact:.6f}")
            print(f"  SBC of bitstrings: DM vs analytical solution  : {sbc_bs_dm_vs_exact:.6f}")
            print(f"  SBC of bitstrings: DM vs FakeVLQ              : {sbc_bs_noisy_vs_dm:.6f}")

        counts_vlq_filtered = {k: v for k, v in counts_vlq.items() if v > limit_for_plotting}
        counts_aer_filtered = {k: v for k, v in counts_aer.items() if v > limit_for_plotting}

        hist_figsize = (max(10, 4 * L), 5)
        # print(f"FakeVLQ (noisy), L={L} (showing counts > {limit_for_plotting})")
        # display(plot_histogram(counts_vlq_filtered, figsize=hist_figsize))
        # print(f"AerSim (noise-free), L={L} (showing counts > {limit_for_plotting})")
        # display(plot_histogram(counts_aer_filtered, figsize=hist_figsize))
else:
    print("RUN_CIRCUITS=False -- skipping main experiment.")
    results = {}

In [19]:
if RUN_SUMMARY and results:
    print("Circuit depths (gate layers after transpilation)")
    print()
    print(f"{'L':>3}  {'nq':>4}  {'Original':>10}  {'VLQ':>10}")
    print("-" * 35)
    for L, rec in sorted(results.items()):
        c = rec["circuit"]
        print(f"{L:>3}  {rec['nq']:>4}  {c['depth_original']:>10}  "
              f"{c['depth_transpiled_VLQ']:>10}")
elif RUN_SUMMARY:
    print("RUN_SUMMARY=True but `results` is empty -- run the sweep first.")
else:
    print("RUN_SUMMARY=False -- skipping depth table.")

Circuit depths (gate layers after transpilation)

  L    nq    Original         VLQ
-----------------------------------
  1     2           7          11
  2     4          12          27
  3     6          17          50
  4     8          22          80


In [ ]:
if RUN_SUMMARY and results:
    # Combined plot of normalized Hamming-weight distributions (noisy, FakeVLQ) across all layers.
    fig, ax = plt.subplots(figsize=(10, 6))
    cmap = plt.get_cmap("viridis")
    n_lines = max(1, len(results))

    for i, (L, rec) in enumerate(sorted(results.items())):
        p = rec["metrics"]["hamming_weight_prob_noisy"]
        ks = list(range(len(p)))
        color = cmap(i / max(1, n_lines - 1))
        ax.plot(ks, p, marker='o', color=color,
                label=f"L={L}  (nq={rec['nq']}, shots={rec['shots']})")

    ax.axvline(x=1, color='grey', linestyle='--', linewidth=1, label='Ideal peak (k=1, noiseless)')
    ax.set_xlabel("Hamming weight k (number of 1s in measured bitstring)")
    ax.set_ylabel("P(k) = fraction of shots with Hamming weight k")
    ax.set_title("Normalized Hamming-weight distribution — FakeVLQ (noisy)\n"
                 "Initial state: single excitation at center qubit — noiseless case has all probability at k=1")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / "hamming_weight_distribution.png", dpi=300)
    plt.show()

    # squared Bhattacharyya coefficient (SBC) vs number of layers — all available metrics
    layers_sorted = sorted(results.keys())
    sbc_aersim = [results[L]["metrics"]["SBC_HW_FakeQVL_noiseless"] for L in layers_sorted]
    sbc_exact_hw = [results[L]["metrics"]["SBC_HW_FakeVLQ_vs_analytical"] for L in layers_sorted]
    sbc_exact_bs = [results[L]["metrics"]["SBC_BS_FakeVLQ_vs_analytical"] for L in layers_sorted]

    # DM-based fidelities may be None for layers above DENSITY_MATRIX_MAX_L -- filter those out per curve.
    layers_dm_vs_exact = [L for L in layers_sorted
                          if results[L]["metrics"].get("SBC_BS_DM_vs_analytical") is not None]
    sbc_bs_dm_vs_exact = [results[L]["metrics"]["SBC_BS_DM_vs_analytical"]
                        for L in layers_dm_vs_exact]
    layers_noisy_vs_dm = [L for L in layers_sorted
                          if results[L]["metrics"].get("SBC_BS_FakeVLQ_vs_DM") is not None]
    sbc_bs_noisy_vs_dm = [results[L]["metrics"]["SBC_BS_FakeVLQ_vs_DM"]
                         for L in layers_noisy_vs_dm]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(layers_sorted, sbc_aersim, marker='s', color='tab:blue', linewidth=2, markersize=8,
            label='FakeVLQ shots vs AerSim shots (HW)')
    ax.plot(layers_sorted, sbc_exact_hw, marker='^', color='tab:red', linewidth=2, markersize=8,
            label='FakeVLQ shots vs exact statevector (HW)')
    ax.plot(layers_sorted, sbc_exact_bs, marker='o', color='tab:green', linewidth=2, markersize=8,
            label='FakeVLQ shots vs exact statevector (bitstring)')
    if layers_dm_vs_exact:
        ax.plot(layers_dm_vs_exact, sbc_bs_dm_vs_exact, marker='x', color='tab:orange',
                linewidth=2, markersize=10,
                label='density-matrix vs exact statevector (bitstring)')
    if layers_noisy_vs_dm:
        ax.plot(layers_noisy_vs_dm, sbc_bs_noisy_vs_dm, marker='D', color='tab:purple',
                linewidth=2, markersize=8,
                label='FakeVLQ shots vs density-matrix (bitstring)')
    for L, sbc in zip(layers_sorted, sbc_exact_bs):
        ax.annotate(f"{sbc:.4f}", (L, sbc), textcoords="offset points", xytext=(0, -15), ha='center', fontsize=8)
    ax.set_xlabel("Number of layers L")
    ax.set_ylabel("squared Bhattacharyya coefficient (SBC)")
    ax.set_title("squared Bhattacharyya coefficient (SBC)")
    ax.set_xticks(layers_sorted)
    # ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower left', fontsize=9)
    plt.tight_layout()
    plt.savefig(RUN_DIR / "SBhattC.png", dpi=300)
    plt.show()

    # N_k/N vs number of layers — one line per Hamming weight k (matches reference plot).
    layers_sorted_nk = sorted(results.keys())
    max_nq = max(rec["nq"] for rec in results.values())
    pk_per_k = {}
    for k in range(max_nq + 1):
        pk_per_k[k] = [results[L]["metrics"]["hamming_weight_prob_noisy"][k]
                       if k < len(results[L]["metrics"]["hamming_weight_prob_noisy"]) else 0.0
                       for L in layers_sorted_nk]

    fig, ax = plt.subplots(figsize=(10, 6))
    cmap_k = plt.get_cmap("tab10")
    for k, pk_values in pk_per_k.items():
        if any(v > 0 for v in pk_values):
            ax.plot(layers_sorted_nk, pk_values, marker='o',
                    color=cmap_k(k % 10), linewidth=2, markersize=6, label=f"k={k}")
    ax.set_xlabel("n (number of layers)")
    ax.set_ylabel("P(k) = N_k / N")
    ax.set_title("Ratio N_k / N vs. number of layers for various k on FakeVLQ")
    ax.set_xticks(layers_sorted_nk)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(RUN_DIR / "GBS_pyramid-BS_Pk-n-k_graph_FakeVLQ.png", dpi=300)
    plt.show()
elif RUN_SUMMARY:
    print("RUN_SUMMARY=True but `results` is empty -- run the sweep first.")
else:
    print("RUN_SUMMARY=False -- skipping summary plots.")

In [ ]:
if RUN_SUMMARY and results:
    # Combined plot of normalized Hamming-weight distributions on the REAL VLQ device.
    # Only layers whose `counts_VLQreal` exists (real-device runs were enabled and succeeded).
    real_layers = sorted(L for L, rec in results.items()
                         if rec.get("counts_VLQreal") is not None)
    if not real_layers:
        print("No real-VLQ runs in `results` -- enable RUN_VLQ_REAL and re-run the sweep cell.")
    else:
        fig, ax = plt.subplots(figsize=(10, 6))
        cmap = plt.get_cmap("viridis")
        n_lines = max(1, len(real_layers))

        for i, L in enumerate(real_layers):
            rec = results[L]
            nq, shots = rec["nq"], rec["shots"]
            hw = hamming_weight_hist(rec["counts_VLQreal"], nq)
            p_hw = [h / shots for h in hw]
            color = cmap(i / max(1, n_lines - 1))
            ax.plot(range(len(p_hw)), p_hw, marker='o', color=color,
                    label=f"L={L}  (nq={nq}, shots={shots})")

        ax.axvline(x=1, color='grey', linestyle='--', linewidth=1, label='Ideal peak (k=1, noiseless)')
        ax.set_xlabel("Hamming weight k (number of 1s in measured bitstring)")
        ax.set_ylabel("P(k) = fraction of shots with Hamming weight k")
        ax.set_title("Normalized Hamming-weight distribution \u2014 real VLQ device\n"
                     "Initial state: single excitation at center qubit \u2014 noiseless case has all probability at k=1")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.savefig(RUN_DIR / "hamming_weight_distribution_VLQreal.png", dpi=300)
        plt.show()
elif RUN_SUMMARY:
    print("RUN_SUMMARY=True but `results` is empty -- run the sweep first.")
else:
    print("RUN_SUMMARY=False -- skipping real-VLQ Hamming-weight plot.")


In [ ]:
# Persist `results` to the run folder created above.
if SAVE_RESULTS and results:
    target = save_run(RUN_DIR, results)
    print(f"Saved {len(results)} record(s) to {target}")
elif SAVE_RESULTS:
    print("SAVE_RESULTS=True but `results` is empty -- nothing to save.")
else:
    print("SAVE_RESULTS=False -- not writing results file.")
